# Demo: PyTorch Lightning vs fastai on the same task

Same small CNN, same MNIST data loaders, 2 epochs. Part A isolates the training layer. Part B shows fastai's pretrained transfer learning.

**Status: written from the official docs, not yet executed. Run it top to bottom and record your own numbers on slide 10.**

Install (GPU optional): `pip install torch torchvision lightning fastai`

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST("data", train=True, download=True, transform=tfm)
valid_ds = datasets.MNIST("data", train=False, download=True, transform=tfm)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
valid_dl = DataLoader(valid_ds, batch_size=256, num_workers=2)

def make_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(32 * 7 * 7, 10),
    )

EPOCHS = 2
results = {}

## Part A1: PyTorch Lightning
You own the model and the step logic. The Trainer owns the loop.

In [ ]:
import lightning as L

class LitCNN(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.net = make_cnn()

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = F.cross_entropy(self.net(x), y)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self.net(x)
        self.log("val_acc", (logits.argmax(1) == y).float().mean(), prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

lit_model = LitCNN()
trainer = L.Trainer(max_epochs=EPOCHS, accelerator="auto", devices=1,
                    logger=False, enable_checkpointing=False)
t0 = time.perf_counter()
trainer.fit(lit_model, train_dataloaders=train_dl, val_dataloaders=valid_dl)
lit_time = time.perf_counter() - t0
lit_acc = trainer.callback_metrics["val_acc"].item()
results["Lightning"] = (lit_time, lit_acc)
print(f"Lightning: {lit_time:.1f}s, val acc {lit_acc:.4f}")

## Part A2: fastai, same model and loaders
fastai wraps plain PyTorch modules and DataLoaders in a `Learner` (the migration path shown in its docs).

In [ ]:
from fastai.vision.all import DataLoaders, Learner, accuracy

dls = DataLoaders(train_dl, valid_dl, device=device)
learn = Learner(dls, make_cnn(), loss_func=F.cross_entropy, metrics=accuracy)
t0 = time.perf_counter()
learn.fit_one_cycle(EPOCHS, 1e-3)
fa_time = time.perf_counter() - t0
fa_acc = float(learn.validate()[1])
results["fastai"] = (fa_time, fa_acc)
print(f"fastai: {fa_time:.1f}s, val acc {fa_acc:.4f}")

## Part A summary
Note the two runs differ in schedule (Adam fixed lr vs one-cycle), so treat accuracy differences as a learning point, not a verdict on the libraries.

In [ ]:
for name, (t, a) in results.items():
    print(f"{name:10s} time={t:6.1f}s  accuracy={a:.4f}")

## Part B: fastai transfer learning (from the official quick start)
Pretrained ResNet34, fine-tuned. Needs a download of the Pets dataset and weights.

In [ ]:
from fastai.vision.all import *

path = untar_data(URLs.PETS) / "images"
def is_cat(x): return x[0].isupper()
pets = ImageDataLoaders.from_name_func(
    path, get_image_files(path), valid_pct=0.2, seed=42,
    label_func=is_cat, item_tfms=Resize(224))
learn2 = vision_learner(pets, resnet34, metrics=error_rate)
learn2.fine_tune(1)

## Your notes (fill in before presenting)
- Training time:
- Accuracy:
- Lines of code you wrote for each library:
- One thing that surprised you: